# Missing RT https://github.com/cal-itp/gtfs-curator/issues/82

In [1]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

# from shared_utils import bq_utils
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data

In [2]:
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, abbrev_month, analysis_month 

In [3]:
@cache
def gcs_pandas():
    return GCSPandas()

In [4]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [5]:
agencies_with_rt = ["Humboldt Transit Authority",
                   "Lake Transit Authority",
                   "Redwood Coast Transit Authority"]

In [17]:
raw_url = "gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_08.parquet"

In [19]:
fct_monthly_route_df = pd.read_parquet(raw_url,
                                        storage_options={"token": credentials.token}).reset_index()

In [20]:
fct_monthly_route_df.sample()

,index,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
491,491,8,2026,2026-08-01,Calaveras Schedule,None,None,None,None,None,Saturday,0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.00,5,1,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [26]:
fct_monthly_route_df2 = fct_monthly_route_df.loc[fct_monthly_route_df.schedule_name.isin(
 ["Lake Schedule"]
)]

In [27]:
fct_monthly_route_df2.sample()

,index,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
5,5,8,2026,2026-08-01,Lake Schedule,None,None,None,None,None,Sunday,0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.00,5,1,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [30]:
fct_monthly_route_df2.filter(like="vp").columns

Index(['vp_name', 'vp_base64_url', 'vp_messages_per_minute', 'n_vp_trips',
       'daily_vp_trips', 'pct_vp_trips', 'pct_vp_routes',
       'daily_vp_extract_duration_minutes', 'n_days_schedule_and_vp_only'],
      dtype='object')

In [28]:
fct_monthly_route_df2.pct_vp_trips.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: pct_vp_trips, dtype: float64

In [32]:
fct_monthly_route_df2.pct_vp_routes.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: pct_vp_routes, dtype: float64

In [31]:
fct_monthly_route_df2.vp_messages_per_minute.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vp_messages_per_minute, dtype: float64